# Eksperyment C - mixed training synthetic + real

Notebook Colab do odpalenia eksperymentu C: synthetic 10k + mala porcja real train/val.

Warianty: 1%, 5%, 10%, 25% real.

Ten notebook zostawiamy jako osobny etap po eksperymencie B.

In [ ]:
# 0. GPU sanity check
# Jesli ta komorka nie przejdzie: Runtime -> Change runtime type -> GPU.
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader
import torch
assert torch.cuda.is_available(), "Brak GPU w runtime Colaba. Wlacz: Runtime -> Change runtime type -> GPU."
print("CUDA:", torch.version.cuda)
print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
# 1. Repo + zaleznosci
# Jesli pracujesz na swoim forku/branchu, zmien REPO_URL i opcjonalnie REPO_BRANCH.
REPO_URL = "https://github.com/milosz-amg/rareplanes-domain-shift.git"
REPO_BRANCH = ""
REPO_DIR = "rareplanes-domain-shift"

import os
from pathlib import Path

if not os.path.isdir(REPO_DIR):
    !git clone {REPO_URL}
%cd {REPO_DIR}
if REPO_BRANCH:
    !git fetch origin {REPO_BRANCH}
    !git checkout {REPO_BRANCH}
!git pull --ff-only || true

required = [
    "src/make_mixed_dataset.py",
    "src/sweep_C_mixed.sh",
]
missing = [p for p in required if not Path(p).exists()]
assert not missing, "Brakuje skryptow eksperymentu C w sklonowanym repo: " + ", ".join(missing)

!pip -q install ultralytics pycocotools pillow matplotlib pandas tabulate

## Dane dla eksperymentu C

Eksperyment C potrzebuje real train/val do mixed training oraz real test do ewaluacji.

In [ ]:
# 2. Adnotacje + real train/test
from pathlib import Path

B = "https://rareplanes-public.s3.amazonaws.com"
for d in [
    "data/real/annotations",
    "data/real/tarballs",
    "data/synthetic/annotations",
    "data/synthetic/images/train",
    "data/real/PS-RGB_tiled",
]:
    Path(d).mkdir(parents=True, exist_ok=True)

for f in ["instances_train_aircraft.json", "instances_test_aircraft.json"]:
    !test -s data/real/annotations/{f} || curl -sf -o data/real/annotations/{f} {B}/real/metadata_annotations/{f}
    !test -s data/synthetic/annotations/{f} || curl -sf -o data/synthetic/annotations/{f} {B}/synthetic/metadata_annotations/{f}

!test -s data/real/tarballs/train.tar.gz || curl -fL -o data/real/tarballs/train.tar.gz {B}/real/tarballs/train/RarePlanes_train_PS-RGB_tiled.tar.gz
!test -s data/real/tarballs/test.tar.gz || curl -fL -o data/real/tarballs/test.tar.gz {B}/real/tarballs/test/RarePlanes_test_PS-RGB_tiled.tar.gz
!tar -xzf data/real/tarballs/train.tar.gz -C data/real/PS-RGB_tiled/
!tar -xzf data/real/tarballs/test.tar.gz -C data/real/PS-RGB_tiled/

real_tiles = len(list(Path("data/real/PS-RGB_tiled/PS-RGB_tiled").glob("*.png")))
print("real tiles:", real_tiles)
assert real_tiles > 7000, "Za malo real tiles - sprawdz pobieranie tarballi."

In [ ]:
# 3. Synthetic 10k przez HTTPS z resume
import os
import urllib.request
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

DST = Path("data/synthetic/images/train")
DST.mkdir(parents=True, exist_ok=True)
files = [l.strip() for l in open("configs/synthetic_10k_train_files.txt")]
files += [l.strip() for l in open("configs/synthetic_10k_val_files.txt")]
files = [f for f in files if f]
print("do pobrania:", len(files))

def fetch(fn):
    dst = DST / fn
    part = DST / (fn + ".part")
    if dst.exists() and dst.stat().st_size > 0:
        return True
    try:
        urllib.request.urlretrieve(f"{B}/synthetic/train/images/{fn}", part)
        os.replace(part, dst)
        return True
    except Exception:
        if part.exists():
            part.unlink()
        return False

ok = 0
with ThreadPoolExecutor(max_workers=32) as ex:
    for i, r in enumerate(ex.map(fetch, files), 1):
        ok += int(r)
        if i % 1000 == 0:
            print(f"  {i}/{len(files)} ok={ok}")

have = len(list(DST.glob("*.png")))
print(f"GOTOWE: ok={ok}/{len(files)}, na dysku={have}")
assert have >= int(len(files) * 0.99), "Za malo synthetic - uruchom komorke ponownie, resume dociagnie reszte."

In [ ]:
# 4. COCO -> YOLO + synthetic_10k + real_aircraft
!python3 src/coco_to_yolo.py --domain synthetic --classes aircraft --val-frac 0.15
!python3 src/coco_to_yolo.py --domain real --classes aircraft --val-frac 0.15
!python3 src/make_subset.py --n-train 10000 --name synthetic_10k

from pathlib import Path
print({
    "synthetic_10k_train": len(list(Path("data/yolo/synthetic_10k/images/train").glob("*.png"))),
    "real_train": len(list(Path("data/yolo/real_aircraft/images/train").glob("*.png"))),
    "real_val": len(list(Path("data/yolo/real_aircraft/images/val").glob("*.png"))),
})

In [ ]:
# 5. Sweep C: 1%, 5%, 10%, 25% real
EPOCHS = 60
BATCH = 64
WORKERS = 8
DEVICE = 0

!EPOCHS={EPOCHS} BATCH={BATCH} WORKERS={WORKERS} DEVICE={DEVICE} bash src/sweep_C_mixed.sh

In [ ]:
# 6. Tabela wynikow C z JSON-ow
import json
from pathlib import Path
import pandas as pd

rows = []
for p in sorted(Path("results/per_size").glob("expC*_ml.json")):
    d = json.load(open(p))
    m = d.get("metrics", {})
    rows.append({
        "run": d.get("name", p.stem),
        "mAP@50": m.get("AP@.5"),
        "mAP@50:95": m.get("AP@[.5:.95]"),
        "AP_S": m.get("AP_small"),
        "AP_M": m.get("AP_medium"),
        "AP_L": m.get("AP_large"),
        "AR@100": m.get("AR@100"),
        "n_det": d.get("n_detections"),
        "file": str(p),
    })

df = pd.DataFrame(rows)
display(df)
Path("results").mkdir(exist_ok=True)
df.to_csv("results/expC_mixed_summary.csv", index=False)
df.to_markdown("results/expC_mixed_summary.md", index=False)
print("zapisano: results/expC_mixed_summary.csv/md")

In [ ]:
# 7. Pobranie wynikow C na komputer
from pathlib import Path
from google.colab import files
import zipfile

targets = []
targets += list(Path("results/per_size").glob("expC*_ml.json"))
targets += list(Path("results/baselines").glob("expC*_ml.json"))
for extra in ["results/expC_mixed_summary.csv", "results/expC_mixed_summary.md"]:
    p = Path(extra)
    if p.exists():
        targets.append(p)

zip_path = Path("expC_mixed_results.zip")
with zipfile.ZipFile(zip_path, "w") as z:
    for p in targets:
        z.write(p, p.as_posix())

print("pliki w paczce:")
for p in targets:
    print(" -", p)
files.download(str(zip_path))